In [29]:
from scapy.all import *
import os
from scapy.all import sniff, wrpcap, IP, TCP, UDP

# Enable Npcap
conf.use_pcap = True

print("Scapy configured successfully!")
print("Npcap enabled:", conf.use_pcap)

Scapy configured successfully!
Npcap enabled: True


In [30]:
#This displays your available network interfaces, such as Wi-Fi and Ethernet.
show_interfaces()

Source   Index  Name                                            MAC                IPv4             IPv6                                   
libpcap  1      Software Loopback Interface 1                   00:00:00:00:00:00  127.0.0.1        ::1                                    
libpcap  10     Microsoft Wi-Fi Direct Virtual Adapter          26:b2:b9:25:26:77  169.254.32.130   fe80::7c9e:45ec:7c54:65b0              
libpcap  15     TAP-Windows Adapter V9                          00:ff:c2:45:96:e7  169.254.46.8     fe80::c2ce:5f14:6fd7:7be6              
libpcap  16     WAN Miniport (IP)                                                                                                          
libpcap  17     WAN Miniport (IPv6)                                                                                                        
libpcap  5      WAN Miniport (Network Monitor)                                                                                             
libpcap  6      Real

In [31]:
print("Starting network traffic capture...")

packets = sniff(
    count=100,
    timeout=30
)

print("Capture completed!")
print("Total packets:", len(packets))

Starting network traffic capture...
Capture completed!
Total packets: 100


- count=100: Capture up to 100 packets.
- timeout=30: Stop after 30 seconds.
- packets: Stores captured packets.
Run Jupyter Notebook as Administrator if Windows permissions prevent packet capture.

In [32]:
# Display packet information
for packet in packets[:10]:
    print(packet.summary())

Ether / IPv6 / TCP 2409:40c1:31d4:32c9:8834:8b5c:9bc6:26d1:60183 > 2603:1063:5:110::365:7ea3:https A / Raw
Ether / IPv6 / TCP 2603:1063:5:110::365:7ea3:https > 2409:40c1:31d4:32c9:8834:8b5c:9bc6:26d1:60183 A
Ether / ARP who has 10.203.56.81 says 10.203.56.17
Ether / ARP is at 24:b2:b9:25:26:77 says 10.203.56.81
Ether / IPv6 / TCP 2409:40c1:31d4:32c9:8834:8b5c:9bc6:26d1:59091 > 64:ff9b::3384:c16b:https PA / Raw
Ether / IPv6 / TCP 2409:40c1:31d4:32c9:8834:8b5c:9bc6:26d1:59091 > 64:ff9b::3384:c16b:https A / Raw
Ether / IPv6 / TCP 2409:40c1:31d4:32c9:8834:8b5c:9bc6:26d1:59091 > 64:ff9b::3384:c16b:https A / Raw
Ether / IPv6 / TCP 2409:40c1:31d4:32c9:8834:8b5c:9bc6:26d1:59091 > 64:ff9b::3384:c16b:https A / Raw
Ether / IPv6 / TCP 2409:40c1:31d4:32c9:8834:8b5c:9bc6:26d1:59091 > 64:ff9b::3384:c16b:https PA / Raw
Ether / IPv6 / TCP 2606:4700:8ca1:d8b7:9a74:ae6:ccf5:ac50:https > 2409:40c1:31d4:32c9:8834:8b5c:9bc6:26d1:59636 PA / Raw


In [33]:
# Extract basic packet details
from scapy.all import sniff, wrpcap, IP, IPv6, TCP, UDP
for packet in packets[:10]:

    print("----------------------")

    if packet.haslayer(IP):

        print("Source IP:", packet[IP].src)
        print("Destination IP:", packet[IP].dst)

    elif packet.haslayer(IPv6):

        print("Source IP:", packet[IPv6].src)
        print("Destination IP:", packet[IPv6].dst)

    if packet.haslayer(TCP):

        print("Protocol: TCP")
        print("Source Port:", packet[TCP].sport)
        print("Destination Port:", packet[TCP].dport)

    elif packet.haslayer(UDP):

        print("Protocol: UDP")
        print("Source Port:", packet[UDP].sport)
        print("Destination Port:", packet[UDP].dport)

----------------------
Source IP: 2409:40c1:31d4:32c9:8834:8b5c:9bc6:26d1
Destination IP: 2603:1063:5:110::365:7ea3
Protocol: TCP
Source Port: 60183
Destination Port: 443
----------------------
Source IP: 2603:1063:5:110::365:7ea3
Destination IP: 2409:40c1:31d4:32c9:8834:8b5c:9bc6:26d1
Protocol: TCP
Source Port: 443
Destination Port: 60183
----------------------
----------------------
----------------------
Source IP: 2409:40c1:31d4:32c9:8834:8b5c:9bc6:26d1
Destination IP: 64:ff9b::3384:c16b
Protocol: TCP
Source Port: 59091
Destination Port: 443
----------------------
Source IP: 2409:40c1:31d4:32c9:8834:8b5c:9bc6:26d1
Destination IP: 64:ff9b::3384:c16b
Protocol: TCP
Source Port: 59091
Destination Port: 443
----------------------
Source IP: 2409:40c1:31d4:32c9:8834:8b5c:9bc6:26d1
Destination IP: 64:ff9b::3384:c16b
Protocol: TCP
Source Port: 59091
Destination Port: 443
----------------------
Source IP: 2409:40c1:31d4:32c9:8834:8b5c:9bc6:26d1
Destination IP: 64:ff9b::3384:c16b
Protocol: T

In [34]:
os.makedirs("../data/processed", exist_ok=True)

wrpcap("../data/processed/live_traffic.pcap", packets)

print("Packets saved successfully!")

Packets saved successfully!


In [35]:
print("Total packets captured:", len(packets))

if len(packets) > 0:
    print("Live network traffic capture successful!")
else:
    print("No packets captured. Check Npcap and permissions.")

Total packets captured: 100
Live network traffic capture successful!


In [36]:
import pandas as pd

# Load generated CSV
df = pd.read_csv("../data/processed/live_flows.csv")

print("Dataset Shape:", df.shape)

print("\nColumn Names:")
print(df.columns.tolist())

print("\nFirst 5 Rows:")
display(df.head())

Dataset Shape: (17, 82)

Column Names:
['src_ip', 'dst_ip', 'src_port', 'dst_port', 'protocol', 'timestamp', 'flow_duration', 'flow_byts_s', 'flow_pkts_s', 'fwd_pkts_s', 'bwd_pkts_s', 'tot_fwd_pkts', 'tot_bwd_pkts', 'totlen_fwd_pkts', 'totlen_bwd_pkts', 'fwd_pkt_len_max', 'fwd_pkt_len_min', 'fwd_pkt_len_mean', 'fwd_pkt_len_std', 'bwd_pkt_len_max', 'bwd_pkt_len_min', 'bwd_pkt_len_mean', 'bwd_pkt_len_std', 'pkt_len_max', 'pkt_len_min', 'pkt_len_mean', 'pkt_len_std', 'pkt_len_var', 'fwd_header_len', 'bwd_header_len', 'fwd_seg_size_min', 'fwd_act_data_pkts', 'flow_iat_mean', 'flow_iat_max', 'flow_iat_min', 'flow_iat_std', 'fwd_iat_tot', 'fwd_iat_max', 'fwd_iat_min', 'fwd_iat_mean', 'fwd_iat_std', 'bwd_iat_tot', 'bwd_iat_max', 'bwd_iat_min', 'bwd_iat_mean', 'bwd_iat_std', 'fwd_psh_flags', 'bwd_psh_flags', 'fwd_urg_flags', 'bwd_urg_flags', 'fin_flag_cnt', 'syn_flag_cnt', 'rst_flag_cnt', 'psh_flag_cnt', 'ack_flag_cnt', 'urg_flag_cnt', 'ece_flag_cnt', 'down_up_ratio', 'pkt_size_avg', 'init_fwd

,src_ip,dst_ip,src_port,dst_port,protocol,timestamp,flow_duration,flow_byts_s,flow_pkts_s,fwd_pkts_s,...,bwd_pkts_b_avg,fwd_blk_rate_avg,bwd_blk_rate_avg,fwd_seg_size_avg,bwd_seg_size_avg,cwr_flag_count,subflow_fwd_pkts,subflow_bwd_pkts,subflow_fwd_byts,subflow_bwd_byts
0,64:ff9b::14b8:af17,2409:40c1:31d4:32c9:8834:8b5c:9bc6:26d1,443,51659,6,2026-11-12 15:25:05,0.411716,38009.210232,48.577175,26.717446,...,5.0,2.245862e+08,0,838.909091,713.444444,0,11,9,9228,6421
1,20.184.175.23,10.203.56.81,443,51658,6,2026-11-12 15:25:05,0.000000,0.000000,0.000000,0.000000,...,0.0,0.000000e+00,0,66.000000,0.000000,0,2,0,132,0
2,10.203.56.81,10.203.56.17,62406,53,6,2026-11-12 15:25:05,0.074464,2175.547916,40.287924,26.858616,...,0.0,0.000000e+00,0,54.000000,54.000000,0,2,1,108,54
3,2409:40c1:31d4:32c9:8834:8b5c:9bc6:26d1,2409:40c1:31d4:32c9::f6,51689,53,6,2026-11-12 15:25:05,0.073949,3002.068993,40.568500,27.045667,...,0.0,0.000000e+00,0,74.000000,74.000000,0,2,1,148,74
4,2409:40c1:31d4:32c9:8834:8b5c:9bc6:26d1,2409:40c1:31d4:32c9::f6,50372,53,6,2026-11-12 15:25:05,0.073687,3012.743089,40.712744,27.141830,...,0.0,0.000000e+00,0,74.000000,74.000000,0,2,1,148,74


In [37]:
import joblib
import pandas as pd
import numpy as np

# Load trained Random Forest model
model = joblib.load("../models/network_model.joblib")

# Load expected feature names
feature_columns = joblib.load("../models/feature_columns.joblib")

print("Model loaded successfully!")
print("Expected features:", len(feature_columns))

Model loaded successfully!
Expected features: 78


In [38]:
live_flows = pd.read_csv(
    "../data/processed/live_flows.csv"
)

print("Live traffic shape:", live_flows.shape)

display(live_flows.head())

Live traffic shape: (17, 82)


,src_ip,dst_ip,src_port,dst_port,protocol,timestamp,flow_duration,flow_byts_s,flow_pkts_s,fwd_pkts_s,...,bwd_pkts_b_avg,fwd_blk_rate_avg,bwd_blk_rate_avg,fwd_seg_size_avg,bwd_seg_size_avg,cwr_flag_count,subflow_fwd_pkts,subflow_bwd_pkts,subflow_fwd_byts,subflow_bwd_byts
0,64:ff9b::14b8:af17,2409:40c1:31d4:32c9:8834:8b5c:9bc6:26d1,443,51659,6,2026-11-12 15:25:05,0.411716,38009.210232,48.577175,26.717446,...,5.0,2.245862e+08,0,838.909091,713.444444,0,11,9,9228,6421
1,20.184.175.23,10.203.56.81,443,51658,6,2026-11-12 15:25:05,0.000000,0.000000,0.000000,0.000000,...,0.0,0.000000e+00,0,66.000000,0.000000,0,2,0,132,0
2,10.203.56.81,10.203.56.17,62406,53,6,2026-11-12 15:25:05,0.074464,2175.547916,40.287924,26.858616,...,0.0,0.000000e+00,0,54.000000,54.000000,0,2,1,108,54
3,2409:40c1:31d4:32c9:8834:8b5c:9bc6:26d1,2409:40c1:31d4:32c9::f6,51689,53,6,2026-11-12 15:25:05,0.073949,3002.068993,40.568500,27.045667,...,0.0,0.000000e+00,0,74.000000,74.000000,0,2,1,148,74
4,2409:40c1:31d4:32c9:8834:8b5c:9bc6:26d1,2409:40c1:31d4:32c9::f6,50372,53,6,2026-11-12 15:25:05,0.073687,3012.743089,40.712744,27.141830,...,0.0,0.000000e+00,0,74.000000,74.000000,0,2,1,148,74


In [39]:
print("Model features:", len(feature_columns))
print("Live traffic columns:", len(live_flows.columns))

print("\nLive traffic columns:")
print(live_flows.columns.tolist())

Model features: 78
Live traffic columns: 82

Live traffic columns:
['src_ip', 'dst_ip', 'src_port', 'dst_port', 'protocol', 'timestamp', 'flow_duration', 'flow_byts_s', 'flow_pkts_s', 'fwd_pkts_s', 'bwd_pkts_s', 'tot_fwd_pkts', 'tot_bwd_pkts', 'totlen_fwd_pkts', 'totlen_bwd_pkts', 'fwd_pkt_len_max', 'fwd_pkt_len_min', 'fwd_pkt_len_mean', 'fwd_pkt_len_std', 'bwd_pkt_len_max', 'bwd_pkt_len_min', 'bwd_pkt_len_mean', 'bwd_pkt_len_std', 'pkt_len_max', 'pkt_len_min', 'pkt_len_mean', 'pkt_len_std', 'pkt_len_var', 'fwd_header_len', 'bwd_header_len', 'fwd_seg_size_min', 'fwd_act_data_pkts', 'flow_iat_mean', 'flow_iat_max', 'flow_iat_min', 'flow_iat_std', 'fwd_iat_tot', 'fwd_iat_max', 'fwd_iat_min', 'fwd_iat_mean', 'fwd_iat_std', 'bwd_iat_tot', 'bwd_iat_max', 'bwd_iat_min', 'bwd_iat_mean', 'bwd_iat_std', 'fwd_psh_flags', 'bwd_psh_flags', 'fwd_urg_flags', 'bwd_urg_flags', 'fin_flag_cnt', 'syn_flag_cnt', 'rst_flag_cnt', 'psh_flag_cnt', 'ack_flag_cnt', 'urg_flag_cnt', 'ece_flag_cnt', 'down_up_ratio

In [40]:
feature_mapping = {
    "Destination Port": "dst_port",
    "Flow Duration": "flow_duration",
    "Total Fwd Packets": "tot_fwd_pkts",
    "Total Backward Packets": "tot_bwd_pkts",
    "Total Length of Fwd Packets": "totlen_fwd_pkts",
    "Total Length of Bwd Packets": "totlen_bwd_pkts",
    "Fwd Packet Length Max": "fwd_pkt_len_max",
    "Fwd Packet Length Min": "fwd_pkt_len_min",
    "Fwd Packet Length Mean": "fwd_pkt_len_mean",
    "Fwd Packet Length Std": "fwd_pkt_len_std",
    "Bwd Packet Length Max": "bwd_pkt_len_max",
    "Bwd Packet Length Min": "bwd_pkt_len_min",
    "Bwd Packet Length Mean": "bwd_pkt_len_mean",
    "Bwd Packet Length Std": "bwd_pkt_len_std",
    "Flow Bytes/s": "flow_byts_s",
    "Flow Packets/s": "flow_pkts_s",
    "Flow IAT Mean": "flow_iat_mean",
    "Flow IAT Std": "flow_iat_std",
    "Flow IAT Max": "flow_iat_max",
    "Flow IAT Min": "flow_iat_min",
    "Fwd IAT Total": "fwd_iat_tot",
    "Fwd IAT Mean": "fwd_iat_mean",
    "Fwd IAT Std": "fwd_iat_std",
    "Fwd IAT Max": "fwd_iat_max",
    "Fwd IAT Min": "fwd_iat_min",
    "Bwd IAT Total": "bwd_iat_tot",
    "Bwd IAT Mean": "bwd_iat_mean",
    "Bwd IAT Std": "bwd_iat_std",
    "Bwd IAT Max": "bwd_iat_max",
    "Bwd IAT Min": "bwd_iat_min",
    "Fwd PSH Flags": "fwd_psh_flags",
    "Bwd PSH Flags": "bwd_psh_flags",
    "Fwd URG Flags": "fwd_urg_flags",
    "Bwd URG Flags": "bwd_urg_flags",
    "Fwd Header Length": "fwd_header_len",
    "Bwd Header Length": "bwd_header_len",
    "Fwd Packets/s": "fwd_pkts_s",
    "Bwd Packets/s": "bwd_pkts_s",
    "Min Packet Length": "pkt_len_min",
    "Max Packet Length": "pkt_len_max",
    "Packet Length Mean": "pkt_len_mean",
    "Packet Length Std": "pkt_len_std",
    "Packet Length Variance": "pkt_len_var",
    "FIN Flag Count": "fin_flag_cnt",
    "SYN Flag Count": "syn_flag_cnt",
    "RST Flag Count": "rst_flag_cnt",
    "PSH Flag Count": "psh_flag_cnt",
    "ACK Flag Count": "ack_flag_cnt",
    "URG Flag Count": "urg_flag_cnt",
    "CWE Flag Count": "cwr_flag_count",
    "ECE Flag Count": "ece_flag_cnt",
    "Down/Up Ratio": "down_up_ratio",
    "Average Packet Size": "pkt_size_avg",
    "Avg Fwd Segment Size": "fwd_seg_size_avg",
    "Avg Bwd Segment Size": "bwd_seg_size_avg",
    "Fwd Header Length.1": "fwd_header_len",
    "Fwd Avg Bytes/Bulk": "fwd_byts_b_avg",
    "Fwd Avg Packets/Bulk": "fwd_pkts_b_avg",
    "Fwd Avg Bulk Rate": "fwd_blk_rate_avg",
    "Bwd Avg Bytes/Bulk": "bwd_byts_b_avg",
    "Bwd Avg Packets/Bulk": "bwd_pkts_b_avg",
    "Bwd Avg Bulk Rate": "bwd_blk_rate_avg",
    "Subflow Fwd Packets": "subflow_fwd_pkts",
    "Subflow Fwd Bytes": "subflow_fwd_byts",
    "Subflow Bwd Packets": "subflow_bwd_pkts",
    "Subflow Bwd Bytes": "subflow_bwd_byts",
    "Init_Win_bytes_forward": "init_fwd_win_byts",
    "Init_Win_bytes_backward": "init_bwd_win_byts",
    "act_data_pkt_fwd": "fwd_act_data_pkts",
    "min_seg_size_forward": "fwd_seg_size_min",
    "Active Mean": "active_mean",
    "Active Std": "active_std",
    "Active Max": "active_max",
    "Active Min": "active_min",
    "Idle Mean": "idle_mean",
    "Idle Std": "idle_std",
    "Idle Max": "idle_max",
    "Idle Min": "idle_min"
}

print("Feature mapping created:", len(feature_mapping))

Feature mapping created: 78


In [41]:
print("feature_columns exists:", "feature_columns" in globals())
print("feature_mapping exists:", "feature_mapping" in globals())
print("live_flows exists:", "live_flows" in globals())
print("pandas loaded:", "pd" in globals())

if "feature_mapping" in globals():
    print("Mapped features:", len(feature_mapping))

if "live_flows" in globals():
    print("Live traffic columns:", len(live_flows.columns))

feature_columns exists: True
feature_mapping exists: True
live_flows exists: True
pandas loaded: True
Mapped features: 78
Live traffic columns: 82


In [42]:
# Check missing features
missing_features = [
    feature for feature in feature_columns
    if feature not in feature_mapping
]

missing_columns = [
    column for column in feature_mapping.values()
    if column not in live_flows.columns
]

print("Missing model features:", missing_features)
print("Missing live columns:", missing_columns)

# Stop if any required feature is missing
if missing_features or missing_columns:
    raise ValueError("Feature mapping is incomplete!")

# Create live feature DataFrame in correct order
live_features = pd.DataFrame(index=live_flows.index)

for feature in feature_columns:
    live_column = feature_mapping[feature]
    live_features[feature] = live_flows[live_column]

print("\nLive features created successfully!")
print("Shape:", live_features.shape)

display(live_features.head())

Missing model features: []
Missing live columns: []

Live features created successfully!
Shape: (17, 78)


,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,act_data_pkt_fwd,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min
0,51659,0.411716,11,9,9228,6421,1354,74,838.909091,578.882694,...,9,0,0,0,0,0,0,0,0,0
1,51658,0.000000,2,0,132,0,66,66,66.000000,0.000000,...,0,20,0,0,0,0,0,0,0,0
2,53,0.074464,2,1,108,54,54,54,54.000000,0.000000,...,0,20,0,0,0,0,0,0,0,0
3,53,0.073949,2,1,148,74,74,74,74.000000,0.000000,...,0,0,0,0,0,0,0,0,0,0
4,53,0.073687,2,1,148,74,74,74,74.000000,0.000000,...,0,0,0,0,0,0,0,0,0,0


## Phase 8: Live Attack Prediction & Uncertainty Analysis

In this phase, we:
1. Validate and clean live features (handling zero-duration flow rates, infinite values, and missing values).
2. Align feature units (converting seconds to microseconds for duration and IAT features to match CIC-IDS2017 training format).
3. Run inference using the trained Random Forest model (`network_model.joblib`).
4. Calculate prediction confidence and uncertainty ($Uncertainty = 1 - \max(P)$).
5. Apply a confidence threshold (0.70) to flag low-confidence flows for security review.
6. Display a structured summary table and save prediction results.

In [ ]:
# Phase 8 - Step 1: Feature Cleaning & Unit Alignment

# Features measured in microseconds in CIC-IDS2017
time_features = [
    "Flow Duration", "Flow IAT Mean", "Flow IAT Std", "Flow IAT Max", "Flow IAT Min",
    "Fwd IAT Total", "Fwd IAT Mean", "Fwd IAT Std", "Fwd IAT Max", "Fwd IAT Min",
    "Bwd IAT Total", "Bwd IAT Mean", "Bwd IAT Std", "Bwd IAT Max", "Bwd IAT Min",
    "Active Mean", "Active Std", "Active Max", "Active Min",
    "Idle Mean", "Idle Std", "Idle Max", "Idle Min"
]

X_live = live_features.copy()

# 1. Convert time features from seconds to microseconds if needed
for col in time_features:
    if col in X_live.columns:
        if X_live[col].dropna().median() < 1000:
            X_live[col] = X_live[col] * 1_000_000.0

# 2. Handle infinite values (from zero-duration division in rates)
X_live = X_live.replace([np.inf, -np.inf], 0.0)

# 3. Handle missing values
X_live = X_live.fillna(0.0)

# 4. Ensure non-negative values
X_live = X_live.clip(lower=0.0)

print("Live features cleaned and aligned:")
print("Shape:", X_live.shape)
print("Missing values:", X_live.isnull().sum().sum())
print("Infinite values:", np.isinf(X_live.values).sum())


In [ ]:
# Phase 8 - Step 2: Model Inference & Probability Estimation

# Predict binary class: 0 = BENIGN, 1 = ATTACK
predictions = model.predict(X_live)

# Predict class probabilities: [P(BENIGN), P(ATTACK)]
probabilities = model.predict_proba(X_live)

print("Predictions array:", predictions)
print("Class probabilities preview:")
print(probabilities[:5])


In [ ]:
# Phase 8 - Step 3: Confidence & Uncertainty Analysis

# Prediction confidence is the maximum class probability
confidence = np.max(probabilities, axis=1)

# Uncertainty is (1 - confidence)
uncertainty = 1.0 - confidence

# Low-confidence threshold: 0.70
confidence_threshold = 0.70

status = [
    "High Confidence" if c >= confidence_threshold else "Low Confidence (Manual Review)"
    for c in confidence
]

print("Confidence scores:", np.round(confidence, 4))
print("Uncertainty scores:", np.round(uncertainty, 4))
print("Status counts:")
print(pd.Series(status).value_counts())


In [ ]:
# Phase 8 - Step 4: Structured Results & Saving

label_map = {0: "BENIGN", 1: "ATTACK"}
proto_map = {6: "TCP", 17: "UDP"}

protocols = [
    proto_map.get(p, str(p)) if p in proto_map else str(p)
    for p in live_flows["protocol"]
] if "protocol" in live_flows.columns else ["Unknown"] * len(live_flows)

results_df = pd.DataFrame({
    "flow_id": range(1, len(live_flows) + 1),
    "src_ip": live_flows["src_ip"] if "src_ip" in live_flows.columns else "N/A",
    "dst_ip": live_flows["dst_ip"] if "dst_ip" in live_flows.columns else "N/A",
    "src_port": live_flows["src_port"] if "src_port" in live_flows.columns else 0,
    "dst_port": live_flows["dst_port"] if "dst_port" in live_flows.columns else 0,
    "protocol": protocols,
    "prediction": predictions,
    "label": [label_map[p] for p in predictions],
    "prob_benign": np.round(probabilities[:, 0], 4),
    "prob_attack": np.round(probabilities[:, 1], 4),
    "confidence": np.round(confidence, 4),
    "uncertainty": np.round(uncertainty, 4),
    "status": status
})

# Save results for FastAPI backend and database
output_path = "../data/processed/live_prediction_results.csv"
results_df.to_csv(output_path, index=False)
print(f"Results saved to: {output_path}")

display(results_df[[
    "flow_id", "src_ip", "dst_ip", "protocol", "label",
    "confidence", "uncertainty", "status"
]])


In [ ]:
# Phase 8 - Step 5: Summary Statistics Report

print("=" * 55)
print("       PHASE 8 LIVE TRAFFIC PREDICTION REPORT       ")
print("=" * 55)
print(f"Total Flows Evaluated  : {len(results_df)}")
print(f"Benign Traffic Flows   : {(predictions == 0).sum()}")
print(f"Attack Traffic Flows   : {(predictions == 1).sum()}")
print(f"High-Confidence Alerts : {status.count('High Confidence')}")
print(f"Low-Confidence Alerts  : {len(status) - status.count('High Confidence')}")
print(f"Average Confidence     : {np.mean(confidence):.4f}")
print(f"Average Uncertainty    : {np.mean(uncertainty):.4f}")
print("=" * 55)
